<a href="https://colab.research.google.com/github/antoniovfonseca/agentic-ai-global-lulc/blob/main/notebooks/MCP_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Environment Setup

Install the official `mcp` (Model Context Protocol) SDK (Software Development Kit) and the `numpy` library for numerical processing.

In [1]:
# 1. Install the Model Context Protocol SDK and numpy
!pip install -q mcp numpy

## 2. Imports and Server Initialization

Import the libraries and initialize the FastMCP server instance.

In [2]:
import numpy as np
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("My Colab MCP")

## 3. Tool Registration

Define a function decorated with `@mcp.tool()`. This example uses `numpy` to calculate statistics (mean and standard deviation) from a list of numbers, demonstrating how to expose Python logic to the MCP server.

In [3]:
import numpy as np

@mcp.tool()
def calculate_statistics(data: list[float]) -> str:
    """
    Computes the statistical mean and standard deviation for a sequence of numbers.

    Parameters
    ----------
    data : list of float
        A sequence of numerical values to be processed.

    Returns
    -------
    str
        A formatted string containing the calculated mean and standard deviation.
    """
    array_data = np.array(data)
    mean_val = np.mean(array_data)
    std_val = np.std(array_data)

    return f"Mean: {mean_val:.2f}, Standard Deviation: {std_val:.2f}"

## 4. Local Tool Verification

Test the registered tool by invoking the function directly with sample data. This ensures the logic behaves as expected before starting the server process.

In [4]:
# 1. Define a sample list of floats for testing (Mock Data)
sample_data = [10.5, 22.3, 33.1, 45.0, 12.8]

# 2. Call the function directly to verify the output
result = calculate_statistics(sample_data)

# 3. Print the formatted result to the console
print(f"Test Result: {result}")

Test Result: Mean: 24.74, Standard Deviation: 12.90


## 5. Running the Server (Threaded Approach)

Since Google Colab controls the main event loop, I execute the MCP server in a separate thread. This prevents the `asyncio` conflict and allows the server to run in the background.

In [6]:
import threading

def start_mcp_server_in_background() -> threading.Thread:
    """
    Starts the MCP server in a separate background thread to avoid blocking Colab.

    Returns
    -------
    threading.Thread
        The daemon thread object running the MCP server.
    """
    def run_mcp_server():
        try:
            mcp.run()
        except Exception as e:
            print(f"\nServer error: {e}")

    # Create the thread
    server_thread = threading.Thread(
        target=run_mcp_server,
        daemon=True
    )

    print("Starting MCP Server in a background thread...")

    # Start the thread
    server_thread.start()

    print("Server is running! You can now continue using other cells.")

    return server_thread

# Call the function to start the server
server_thread = start_mcp_server_in_background()


Starting MCP Server in a background thread...
Server is running! You can now continue using other cells.

Server error: I/O operation on closed file


## 6. Installing and Configuring the Gemini LLM

Here I install the official Gemini SDK to act as the brain of our Agentic AI.

In [7]:
!pip install -q google-genai

In [8]:
from google import genai
from google.colab import userdata

# 1. Retrieve the API key securely from Colab's secret manager
api_key = userdata.get('GEMINI_API_KEY')

# 2. Initialize the connection to the Gemini LLM
gemini_client = genai.Client(api_key=api_key)
print("Gemini LLM successfully connected!")

Gemini LLM successfully connected!


## 7. The Agentic Loop: Triggering Function Calls

Here I create a simple orchestrator. I send a natural language prompt to Gemini and provide our `calculate_statistics` tool. We will observe how the LLM decides to format a tool call instead of generating regular text.

In [12]:
import time
from datetime import datetime

# ==========================================
# 1. USER INPUT (Modify this prompt!)
# ==========================================
user_prompt = "Calculate the mean and standard deviation for: [10.5, 22.3, 33.1, 45.0, 12.8]"

# ==========================================
# 2. HELPER FUNCTIONS & SETUP
# ==========================================
def log_event(event_name: str) -> None:
    """
    Logs a given event string to the standard output with a timestamp prefix.

    Parameters
    ----------
    event_name : str
        The descriptive string representing the event or action to be logged.

    Returns
    -------
    None
        This function does not return any value.
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {event_name}")

# --- MITIGATION: GLOBAL RATE LIMIT COOLDOWN ---
# Protects against RPM exhaustion if this script/cell is executed multiple times in a row.
print("Applying global network cooldown (5 seconds) before starting...")
time.sleep(5)

# ==========================================
# 3. AGENTIC LOOP EXECUTION
# ==========================================
try:
    print("==================================================")
    log_event("STARTING AGENTIC LOOP")

    # 1. First Turn: Forcing tool usage
    log_event("Host -> LLM: Sending prompt and enforcing tool protocol")
    response = gemini_client.models.generate_content(
        model='gemini-2.5-flash-lite', #gemini-3.1-flash-lite-preview
        contents=user_prompt,
        config={
            "tools": [calculate_statistics],
            "tool_config": {"function_calling_config": {"mode": "ANY"}}
        }
    )

    # 2. Protocol Check
    if response.function_calls:
        tool_call = response.function_calls[0]
        log_event(f"LLM -> Host: Tool request detected ('{tool_call.name}')")

        # 3. Server Execution
        log_event("Host -> Server: Executing local function")
        tool_result = calculate_statistics(**tool_call.args)
        log_event(f"Server -> Host: Execution complete. Raw result: {tool_result}")

        # --- MITIGATION: INTERNAL RATE LIMIT COOLDOWN ---
        log_event("Host: Applying safety cooldown to prevent internal RPM limits...")
        time.sleep(5)

        # 4. Final Synthesis
        log_event("Host -> LLM: Sending exact tool data for final synthesis")
        final_response = gemini_client.models.generate_content(
            model='gemini-2.5-flash-lite', #gemini-3.1-flash-lite-preview
            contents=[
                user_prompt,
                response.candidates[0].content,
                {
                    "role": "user",
                    "parts": [{
                        "function_response": {
                            "name": tool_call.name,
                            "response": {"output": str(tool_result)}
                        }
                    }]
                }
            ]
        )

        log_event("LLM -> Host: Final response generated")
        print("==================================================\n")

        print("--- Final Agent Response (via MCP) ---")
        print(final_response.text)

    else:
        log_event("Protocol Failure: No tool triggered.")

except Exception as e:
    print(f"Execution error: {e}")


Applying global network cooldown (5 seconds) before starting...
[00:06:47] STARTING AGENTIC LOOP
[00:06:47] Host -> LLM: Sending prompt and enforcing tool protocol
[00:06:56] LLM -> Host: Tool request detected ('calculate_statistics')
[00:06:56] Host -> Server: Executing local function
[00:06:56] Server -> Host: Execution complete. Raw result: Mean: 24.74, Standard Deviation: 12.90
[00:06:56] Host: Applying safety cooldown to prevent internal RPM limits...
[00:07:01] Host -> LLM: Sending exact tool data for final synthesis
[00:07:02] LLM -> Host: Final response generated

--- Final Agent Response (via MCP) ---
The mean for the given dataset [10.5, 22.3, 33.1, 45.0, 12.8] is 24.74, and the standard deviation is 12.90.


In [ ]:
# List all models available for your API Key to avoid 404 errors
print("Listing available models...")
for model in gemini_client.models.list():
    print(f"Model ID: {model.name} - Supported Actions: {model.supported_actions}")

Listing available models...
Model ID: models/gemini-2.5-flash - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.5-pro - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-001 - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-lite-001 - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-lite - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.5-flash-preview-tts - Supported Actions: ['countTokens', 'generateContent']
Model ID: models/gemini-2.5-pro-

In [13]:
# List only models that support text generation (Agentic capabilities)
print("Listing available text-out models...")

for model in gemini_client.models.list():
    # Filter: Check if the model supports generating content (text/reasoning)
    if 'generateContent' in model.supported_actions:
        print(f"Model ID: {model.name} - Supported Actions: {model.supported_actions}")

Listing available text-out models...
Model ID: models/gemini-2.5-flash - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.5-pro - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-001 - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-lite-001 - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-lite - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.5-flash-preview-tts - Supported Actions: ['countTokens', 'generateContent']
Model ID: models/gemini